# 3. Data Preparation: Ekstraksi Fitur & Pembagian Data Berdasarkan Kelompok

Tahap ini mengekstrak fitur visual dan menghindari kebocoran foto dari kelompok pengambilan yang sama:
1. **Split group-wise:** Kelompok 2 (15 Rapi, 15 Berantakan) menjadi test; kelompok lain menjadi data pengembangan.
2. **Ekstraksi fitur:** Gambar diubah ke ukuran 224 ? 224 untuk fitur RGB/HSV, grayscale, tepi, kuadran, dan histogram gradien lokal HOG.
3. **Metadata dimensi gambar** disimpan untuk audit, tetapi tidak menjadi masukan model.
4. **Standardisasi fitur** dipelajari ulang di dalam setiap fold validasi dan pada data pengembangan penuh untuk model akhir.

Dengan tujuh kelompok, uji akhir hanya mewakili satu kelompok; hasil perlu dianggap sebagai estimasi awal.

In [ ]:
import sys
import re
from pathlib import Path
import pandas as pd
from PIL import Image


def find_project_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / Path("src/1-project/proyek_sain_data/src/10-project_citra")):
            if (candidate / "3-data_preparation").is_dir() and (candidate / "6-deployment").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Folder proyek 10-project_citra tidak ditemukan dari direktori kerja.")


PROJECT_DIR = find_project_dir()
DATASET_DIR = PROJECT_DIR.parents[1] / "data"
if not all((DATASET_DIR / name).is_dir() for name in ("RAPI", "BERANTAKAN")):
    raise FileNotFoundError(f"Folder kelas dataset tidak lengkap: {DATASET_DIR}")
sys.path.insert(0, str(PROJECT_DIR))
from feature_extraction import FEATURE_COLUMNS, extract_image_features

PREPARATION_DIR = PROJECT_DIR / "3-data_preparation"
PROCESSED_DIR = PREPARATION_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

---
## 3.1 Ekstraksi Fitur Citra Multi-Domain


In [ ]:
records = []
review_candidates = []
for class_folder, label_id in (("RAPI", 1), ("BERANTAKAN", 0)):
    for image_path in sorted((DATASET_DIR / class_folder).rglob("*")):
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        with Image.open(image_path) as image:
            features = extract_image_features(image)
        filename_hint = image_path.stem.casefold()
        if (label_id == 1 and "berantakan" in filename_hint) or (label_id == 0 and "rapi" in filename_hint):
            review_candidates.append({
                "rel_path": image_path.relative_to(DATASET_DIR).as_posix(),
                "folder_label": "Rapi" if label_id == 1 else "Berantakan",
                "filename_hint": "Berantakan" if label_id == 1 else "Rapi",
            })
        group_match = re.search(r"\d+", image_path.parent.name)
        if group_match is None:
            raise ValueError(f"Nama kelompok tidak memiliki nomor: {image_path.parent.name}")
        records.append({
            **features,
            "filename": image_path.name,
            "rel_path": image_path.relative_to(DATASET_DIR).as_posix(),
            "kelompok": f"kelompok {group_match.group(0)}",
            "class": "Rapi" if label_id == 1 else "Berantakan",
            "label": label_id,
        })

df_all = pd.DataFrame(records)
if df_all.empty or set(df_all["label"]) != {0, 1}:
    raise ValueError("Data harus berisi gambar dari kedua kelas.")
if df_all["rel_path"].duplicated().any():
    raise ValueError("Dataset berisi path gambar duplikat.")
pd.DataFrame(review_candidates, columns=["rel_path", "folder_label", "filename_hint"]).to_csv(
    PROCESSED_DIR / "label_review_candidates.csv", index=False
)
if review_candidates:
    print(f"Tinjau {len(review_candidates)} kandidat label di label_review_candidates.csv; label tidak diubah otomatis.")
print(f"Total citra berhasil diekstrak: {len(df_all)}")
display(df_all.head(5))

---
## 3.2 Group-wise Train-Test Split

Kelompok 2 (seimbang 15/15 antar kelas) menjadi test akhir. Seluruh foto dari kelompok yang sama tetap pada split yang sama. Kelompok lain menjadi data pengembangan; model selection di tahap Modeling menggunakan StratifiedGroupKFold pada kelompok-kelompok tersebut.

In [ ]:
TEST_GROUP = "kelompok 2"
if TEST_GROUP not in set(df_all["kelompok"]):
    raise ValueError(f"Kelompok uji tidak ditemukan: {TEST_GROUP}")

df_all["split"] = "train"
df_all.loc[df_all["kelompok"] == TEST_GROUP, "split"] = "test"

split_groups = {split: set(part["kelompok"]) for split, part in df_all.groupby("split")}
if split_groups["train"] & split_groups["test"]:
    raise ValueError("Kelompok yang sama muncul pada train dan test.")
for split, part in df_all.groupby("split"):
    if set(part["label"]) != {0, 1}:
        raise ValueError(f"Split {split} tidak memuat kedua kelas.")

feature_cols = FEATURE_COLUMNS.copy()
manifest = df_all.copy()
manifest.to_csv(PROCESSED_DIR / "dataset_manifest.csv", index=False)
manifest.to_csv(PREPARATION_DIR / "dataset_manifest.csv", index=False)
for split in ("train", "test"):
    manifest.loc[manifest["split"] == split].to_csv(
        PROCESSED_DIR / f"{split}_features.csv", index=False
    )

train_df = manifest.loc[manifest["split"] == "train"]
test_df = manifest.loc[manifest["split"] == "test"]
print(f"Total sampel: {len(manifest)}")
print(manifest.groupby(["split", "class"]).size().unstack(fill_value=0))
print(f"Kelompok uji: {TEST_GROUP}; seluruh kelompok lain masuk data pengembangan.")
print(f"Train: {len(train_df)}, test: {len(test_df)}")